# AI-Powered Flood Risk Prediction
# Random Forest
## Ghale

Trains a Random Forest model to classify flood risk at Murray Bridge using real historical river level data.

In [1]:
!pip install pandas numpy scikit-learn joblib -q


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\kingn\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1. Load real data

In [2]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, matthews_corrcoef, classification_report
import joblib

HERE = Path.cwd()
REPO = HERE if (HERE / "data").exists() else HERE.parent
DATA = REPO / "data"
NOTEBOOKS = REPO / "notebooks"

df = pd.read_csv(DATA / "murray_bridge_river_level_historical.csv", skiprows=4,
                 names=["datetime", "water_level_m", "conductivity", "water_temp_c"])
df["datetime"] = pd.to_datetime(df["datetime"], format="%H:%M:%S %d/%m/%Y", errors="coerce")
df = df.dropna(subset=["datetime", "water_level_m"]).sort_values("datetime").reset_index(drop=True)

print("Rows:", len(df))
print("Date range:", df["datetime"].min(), "to", df["datetime"].max())
df.head()

Rows: 6364
Date range: 2009-01-08 09:00:00 to 2026-07-02 09:00:00


,datetime,water_level_m,conductivity,water_temp_c
0,2009-01-08 09:00:00,-0.584,680.492,22.788
1,2009-01-09 09:00:00,-0.640,682.482,22.719
2,2009-01-10 09:00:00,-0.662,685.370,22.657
3,2009-01-11 09:00:00,-0.637,691.286,22.730
4,2009-01-12 09:00:00,-0.649,691.175,22.737


## 2. Feature engineering

In [3]:
df["level_lag1"] = df["water_level_m"].shift(1)
df["level_lag2"] = df["water_level_m"].shift(2)
df["level_roll7"] = df["water_level_m"].shift(1).rolling(7).mean()
df["level_change3"] = df["water_level_m"].shift(1) - df["water_level_m"].shift(4)
df = df.dropna(subset=["level_lag1", "level_lag2", "level_roll7", "level_change3"])

threshold = df["water_level_m"].quantile(0.80)
df["high_risk"] = (df["water_level_m"] >= threshold).astype(int)

print("Risk threshold (m):", round(threshold, 3))
df["high_risk"].value_counts()

Risk threshold (m): 0.806


high_risk
0    5083
1    1274
Name: count, dtype: int64

## 3. Train/test split

In [4]:
features = ["level_lag1", "level_lag2", "level_roll7", "level_change3"]
X = df[features]
y = df["high_risk"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 4. Train Random Forest

In [5]:
model_rf = RandomForestClassifier(n_estimators=400, min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1)
model_rf.fit(X_train, y_train)
pred_rf = model_rf.predict(X_test)
f1_rf = f1_score(y_test, pred_rf)
mcc_rf = matthews_corrcoef(y_test, pred_rf)
print(f"F1: {f1_rf:.3f}  MCC: {mcc_rf:.3f}")
print(classification_report(y_test, pred_rf))

F1: 0.795  MCC: 0.746
              precision    recall  f1-score   support

           0       0.94      0.96      0.95      1017
           1       0.81      0.78      0.80       255

    accuracy                           0.92      1272
   macro avg       0.88      0.87      0.87      1272
weighted avg       0.92      0.92      0.92      1272



## 5. Save trained model

In [6]:
joblib.dump(model_rf, NOTEBOOKS / "Random_Forest.joblib")
print("Model saved.")

Model saved.
